# 08_05 Decentral markdown and spoilage history features

**Batch statement (pre-registered before the refit).** Store staff reduce prices of
meat approaching its date — the `L Dezentrale Preisreduzierungen` / `F Dezentrale
Aktionen` codes of the abschriften event table. These are same-day store decisions,
invisible to the model at forecast time: on eval rows with such an event, demand runs
at **1.61×** the series' weekday mean while the pre-batch mainline forecasts only
**0.59×** of the actual. The event itself is unknowable 1–7 days ahead (an oracle
level-fix on those rows *hurts*, `09_01`), so the batch ships **propensity and
response history**, all strictly pre-origin:

| feature | mechanism (verified 2026-08-18, section 1) |
|---|---|
| `decentral_markdowns_last_28d` | P(markdown day in forecast week) climbs 1.3% → 37.7% from 0 to 8+ prior events |
| `days_since_last_decentral_markdown` | recency companion to the count |
| `series_markdown_rate` | lifetime rate, EB-shrunk (prior 56 active days): separates the ~78% of series that never mark down |
| `article_markdown_lift` | per-article lift on markdown days, EB-shrunk (prior 24 obs); split-half corr **0.78** — stable enough to learn |
| `markdown_expected_lift` | lift × recent propensity × network weekday tilt (Sat +39% vs Mon) — the precomputed interaction, `06_02` lesson |
| `spoilage_days_last_28d` | `Q Bruch/Verderb` days; raises markdown propensity monotonically *beyond* markdown history (the old spoilage features were silently empty — pipeline bug, not a dead mechanism) |

**Expected effect:** the affected rows are ~1% of rows / ~1.7% of volume at 41%
underforecast, plus the propensity channel on quiet rows: **−0.3 to −1.5 pp row
WAPE**, hard-bounded by the ~3.7 pp perfect-information oracle. **Guardrail:** the
underforecast on markdown rows must shrink *without* pushing global bias off ~0
(baseline −0.25%).

Baseline: the `08_04` mainline (row 56.92%, bias −0.25%). `FEATURE_BUILDER_VERSION`
2026-08-16.3 → 2026-08-18.1, all 72 partitions rebuilt, two-stage refit only.

**Outcome (spoiler for the reader): the batch was REVERTED** — see the verdict at
the end. The mainline result files were restored; this run's artifacts are
preserved in `reports/results/backup_08_05_markdown_null/`, the feature code was
removed with the six columns recorded in `REMOVED_FEATURE_COLUMNS`, and the
builder version moved on to 2026-08-18.2 (same logic as 2026-08-16.3, new version
so no cached partition claims a lineage the code no longer guarantees).


In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.results import result_path

pd.set_option('display.max_columns', 40)

design = load_benchmark_design()
con = duckdb.connect()
con.execute('PRAGMA threads=4')

# after the revert, the standard result paths hold the restored 08_04 mainline
# ("before"); this batch's run is preserved under backup_08_05_markdown_null
BATCH_RUN = ROOT / 'reports/results/backup_08_05_markdown_null'

for name, path in (('before', result_path(TWO_STAGE_MODEL_NAME, design)),
                   ('after', BATCH_RUN / 'forecasts_two_stage.csv')):
    con.execute(f'''
        CREATE OR REPLACE TEMP TABLE {name} AS
        SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
               sourcing_group, category_id::INTEGER AS category_id,
               CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
               actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
               EXTRACT(ISODOW FROM CAST(period AS DATE)) AS weekday
        FROM read_csv_auto(?) WHERE is_active
    ''', [str(path)])

# decentral markdown days (L/F) on the eval window, for the targeted-row audit
con.execute('''
    CREATE OR REPLACE TEMP TABLE markdown_days AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(DATE AS DATE) AS period
    FROM read_parquet(?)
    WHERE ABSCHRIFT_ART IN ('L', 'F') AND WGR_ID IN (890, 900)
    GROUP BY 1, 2, 3
''', [str(ROOT / 'data/interim/abschriften/abschriften_year_2026.parquet')])

sanity = con.execute('''
    SELECT (SELECT COUNT(*) FROM before) AS before_rows,
           (SELECT COUNT(*) FROM after) AS after_rows
''').fetchdf()
assert int(sanity.before_rows[0]) == int(sanity.after_rows[0])
display(sanity.style.format('{:,.0f}').hide(axis='index'))


before_rows,after_rows
"2,498,967","2,498,967"


## 1 · The mechanism, measured on the pre-batch baseline

Recorded here so the batch is judged against a stated expectation: how demand and
the *previous* model behave on decentral-markdown days.


In [2]:
mechanism = con.execute('''
    WITH wd AS (SELECT ARTIKEL_ID, MARKT_ID, weekday, AVG(actual) AS mu
                FROM before GROUP BY 1, 2, 3)
    SELECT (m.ARTIKEL_ID IS NOT NULL) AS markdown_day, COUNT(*) AS n_rows,
           SUM(b.actual) / SUM(wd.mu) AS demand_vs_own_weekday_mean,
           SUM(b.forecast) / NULLIF(SUM(b.actual), 0) AS forecast_over_actual,
           SUM(ABS(b.forecast - b.actual)) / NULLIF(SUM(b.actual), 0) AS wape
    FROM before b
    JOIN wd USING (ARTIKEL_ID, MARKT_ID, weekday)
    LEFT JOIN markdown_days m USING (ARTIKEL_ID, MARKT_ID, period)
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(mechanism.style.format({
    'n_rows': '{:,.0f}', 'demand_vs_own_weekday_mean': '{:.2f}',
    'forecast_over_actual': '{:.3f}', 'wape': '{:.2%}'}).hide(axis='index'))


markdown_day,n_rows,demand_vs_own_weekday_mean,forecast_over_actual,wape
False,"2,473,531",0.99,1.006,56.84%
True,"25,436",1.61,0.593,60.66%


## 2 · Results against the backed-up baseline

Standard batch scoring: pooled WAPE at row / article-store-week / article-day /
article-week grains plus relative bias, before vs after, and the per-origin
distribution of the row-level change.


In [3]:
GRAINS = {
    'row': None,
    'article-store-week': 'ARTIKEL_ID, MARKT_ID, origin',
    'article-day': 'ARTIKEL_ID, origin, period',
    'article-week': 'ARTIKEL_ID, origin',
}


def score(table):
    out = {}
    for grain, keys in GRAINS.items():
        inner = (f'SELECT actual, forecast FROM {table}' if keys is None else
                 f'SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                 f'FROM {table} GROUP BY {keys}')
        wape, bias = con.execute(
            f'SELECT SUM(ABS(forecast - actual)) / SUM(actual), '
            f'SUM(forecast - actual) / SUM(actual) FROM ({inner})').fetchone()
        out[grain] = wape
        if keys is None:
            out['bias'] = bias
    return out


results = pd.DataFrame([
    {'run': '08_04 baseline', **score('before')},
    {'run': '08_05 markdown features', **score('after')},
])
delta = {'run': 'delta (pp)',
         **{g: (results.iloc[1][g] - results.iloc[0][g]) * 100 for g in GRAINS},
         'bias': (results.iloc[1].bias - results.iloc[0].bias) * 100}
display(pd.concat([results, pd.DataFrame([delta])], ignore_index=True).style.format(
    {g: '{:.2%}' for g in GRAINS} | {'bias': '{:+.2%}'},
    subset=pd.IndexSlice[[0, 1], :]).format(
    {g: '{:+.2f}' for g in GRAINS} | {'bias': '{:+.2f}'},
    subset=pd.IndexSlice[[2], :]).hide(axis='index'))

per_origin = con.execute('''
    WITH b AS (SELECT origin, SUM(ABS(forecast - actual)) / SUM(actual) AS wape
               FROM before GROUP BY 1),
    a AS (SELECT origin, SUM(ABS(forecast - actual)) / SUM(actual) AS wape
          FROM after GROUP BY 1)
    SELECT b.origin, b.wape AS before_wape, a.wape AS after_wape,
           (a.wape - b.wape) * 100 AS delta_pp
    FROM b JOIN a USING (origin) ORDER BY origin
''').fetchdf()
improved = int((per_origin.delta_pp < 0).sum())
print(f'{improved}/20 origins improved; '
      f'per-origin delta {per_origin.delta_pp.min():+.2f} .. '
      f'{per_origin.delta_pp.max():+.2f} pp')
display(per_origin.style.format({'before_wape': '{:.2%}', 'after_wape': '{:.2%}',
                                 'delta_pp': '{:+.2f}'}).hide(axis='index'))


run,row,bias,article-store-week,article-day,article-week
08_04 baseline,56.92%,-0.25%,36.40%,23.41%,19.88%
08_05 markdown features,57.00%,+0.04%,36.56%,23.47%,20.01%
delta (pp),+0.08,+0.29,+0.16,+0.06,+0.14


7/20 origins improved; per-origin delta -1.05 .. +0.96 pp


origin,before_wape,after_wape,delta_pp
2026-03-02 00:00:00,54.92%,55.87%,+0.96
2026-03-09 00:00:00,59.60%,59.73%,+0.13
2026-03-16 00:00:00,55.93%,55.91%,-0.01
2026-03-23 00:00:00,59.93%,60.47%,+0.54
2026-03-30 00:00:00,48.49%,48.31%,-0.18
2026-04-06 00:00:00,58.57%,57.78%,-0.79
2026-04-13 00:00:00,67.90%,68.86%,+0.96
2026-04-20 00:00:00,58.89%,58.32%,-0.57
2026-04-27 00:00:00,57.05%,56.00%,-1.05
2026-05-04 00:00:00,55.03%,55.08%,+0.05


## 3 · Did it hit the targeted rows?

The batch exists to shrink the 41% underforecast on realized markdown days without
buying it with bias elsewhere. Same audit as section 1, before vs after, plus the
non-markdown rows as the control group.


In [4]:
targeted = con.execute('''
    SELECT (m.ARTIKEL_ID IS NOT NULL) AS markdown_day, run,
           SUM(forecast) / NULLIF(SUM(actual), 0) AS forecast_over_actual,
           SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS wape
    FROM (
        SELECT *, 'before' AS run FROM before
        UNION ALL SELECT *, 'after' AS run FROM after
    )
    LEFT JOIN markdown_days m USING (ARTIKEL_ID, MARKT_ID, period)
    GROUP BY 1, 2 ORDER BY 1, 2 DESC
''').fetchdf()
display(targeted.style.format({'forecast_over_actual': '{:.3f}',
                               'wape': '{:.2%}'}).hide(axis='index'))


markdown_day,run,forecast_over_actual,wape
False,before,1.006,56.84%
False,after,1.008,56.93%
True,before,0.593,60.66%
True,after,0.598,60.67%


## 4 · Are the features actually used?

Mean gain-share rank of the six new columns within each stage, across the five
refits (rank 1 = the stage's most used feature).


In [5]:
importance = pd.concat([
    pd.read_csv(result_path(TWO_STAGE_MODEL_NAME, design,
                            artifact='feature_importance')).assign(run='before'),
    pd.read_csv(BATCH_RUN / 'feature_importance_two_stage.csv').assign(run='after'),
])
importance['rank'] = importance.groupby(['run', 'evaluation_origin', 'stage'])[
    'gain'].rank(ascending=False)
NEW_FEATURES = ['decentral_markdowns_last_28d', 'days_since_last_decentral_markdown',
                'series_markdown_rate', 'article_markdown_lift',
                'markdown_expected_lift', 'spoilage_days_last_28d']
summary = (importance[importance.run.eq('after')
                      & importance.feature.isin(NEW_FEATURES)]
           .groupby(['stage', 'feature'])
           .agg(mean_rank=('rank', 'mean'), gain_share=('gain_share', 'mean'))
           .sort_values(['stage', 'mean_rank']))
n_features = importance[importance.run.eq('after')].groupby('stage')[
    'feature'].nunique()
print('features per stage:', dict(n_features))
display(summary.style.format({'mean_rank': '{:.1f}', 'gain_share': '{:.2%}'}))


features per stage: {'occurrence': np.int64(64), 'positive_quantity': np.int64(64)}


**Reading the results against the pre-registered expectation.**

- **Global effect: a null, slightly negative.** Row WAPE 56.92% → 57.00%
  (+0.08 pp, within the ±0.07 pp single-seed spread measured in `10_03`), and every
  aggregated grain moved +0.06 to +0.16 pp. Only 7/20 origins improved, with
  per-origin swings (−1.05 to +0.96 pp) that look like refit reshuffling, not a
  mechanism. Bias −0.25% → +0.04% — nominally closer to zero, but of the same
  order as seed noise and not bought by the targeted rows.
- **The targeted rows did not move — the decisive failure.** On realized
  markdown days the forecast/actual ratio went 0.593 → 0.598 and their WAPE
  60.66% → 60.67%. The batch's entire purpose was to close this gap; it closed
  half a point of forty.
- **The boosters agree: they left the features on the shelf.** Best rank is
  `article_markdown_lift` at 17/64 in the quantity stage with 0.53% gain share;
  the other five sit at ranks 42–63 with ≤0.05%. For comparison, the analogous
  `action_expected_lift` reached rank ~3 with 12.7% in `08_04`.

**Why a verified mechanism still failed — the dilution arithmetic.** Every input
check passed (demand really is 1.61× on markdown days; propensity really climbs
29-fold; per-article lift really is stable at split-half r = 0.78). But the event
is a *same-day store decision*: even in the top propensity bucket the chance that
any given horizon day is a markdown day is ~8%, so the honest expected lift on a
quiet-looking day is ~0.18× against the ~0.6× correction the actual event days
need. Spreading that average over ~12 quiet days per event day costs on the 92%
of days without an event roughly what it earns on the 8% with one — which is
exactly what the `09_01` oracle audit predicted when the *oracle level fix* on
these rows made WAPE worse. The trees, seeing both sides of that trade in
training, rationally declined it. The mechanism is real; it is just not
deliverable from history — only a same-day data feed of decentral reductions
(a data-collection investment, not a modelling one) can unlock the ~3.7 pp bound.


## Verdict: REVERTED

Judged against the pre-registered success criterion — shrink the markdown-day
underforecast without a global bias cost — the batch fails: the underforecast is
unchanged, the global WAPE is flat-to-worse at every grain, and the features are
unused. Following the `08_03` precedent for null batches:

- mainline result files restored from the pre-batch backup (the `08_04` run
  remains the adopted mainline: row 56.92%, bias −0.25%);
- this run's artifacts preserved in `reports/results/backup_08_05_markdown_null/`;
- the six feature columns and their SQL removed from `builder.py`; the names are
  recorded in `REMOVED_FEATURE_COLUMNS` with the failure rationale;
- `FEATURE_BUILDER_VERSION` advanced to 2026-08-18.2 (revert marker) so cache
  lineage stays truthful; full test suite green (92 tests).

**What survives the revert:** the mechanism audit in section 1 stands and matters
for the thesis — decentral markdowns are a real, quantified blind spot
(~1% of rows at 41% underforecast, bounded at ~3.7 pp with perfect knowledge),
and this batch establishes *empirically* that the gap cannot be closed from
historical data alone. That converts "we did not model store markdowns" from a
limitation into a measured negative result with a concrete remedy (a same-day
markdown feed) — a stronger statement than an untested feature idea would have
been. It also closes the loop on the spoilage story: the `Q` data is real
(`10_01`-session correction), its propensity signal is real, and its predictive
value is nil for the same dilution reason.
